#### Gerar dados

In [10]:
import igraph as ig
import numpy as np
import pandas as pd
from scipy.sparse.linalg import eigsh
from itertools import product
import semopy

from joblib import Parallel, delayed

In [11]:
def generate_logits(R: int, noise_type: str):
    if noise_type == "uniform": raw = np.random.uniform(0, 1, size=R)
    elif noise_type == "chi_square": raw = np.random.chisquare(df=2, size=R)
    elif noise_type == "exponential": raw = np.random.exponential(scale=1.0, size=R)
    elif noise_type == "laplace": raw = np.random.laplace(loc=0, scale=1, size=R)
    elif noise_type == "logistic": raw = np.random.logistic(loc=0, scale=1, size=R)
    elif noise_type == "student_t": raw = np.random.standard_t(df=3, size=R)
    elif noise_type == "F": raw = np.random.f(dfnum=5, dfden=10, size=R)
    elif noise_type == "gaussian": raw = np.random.normal(loc=0, scale=1, size=R)
    else: 
        raise ValueError("Ruído desconhecido")
    return (raw - np.mean(raw))/np.std(raw)


def gerar_estrutura_causal_logits(R, tipo_ruido, estrutura_causal):
    def logistic(vetor): 
        return 1 / (1 + np.exp(-vetor))
    
    ruido_A = generate_logits(R, tipo_ruido)
    ruido_B = generate_logits(R, tipo_ruido)

    if estrutura_causal == {"A->B"}:
        A = ruido_A
        B = 0.5 * A + ruido_B
        return logistic(A), logistic(B)
    
    elif estrutura_causal == {"A<->B"}:
        ruido_A = generate_logits(R, tipo_ruido)
        ruido_B = generate_logits(R, tipo_ruido)

        A = ruido_A
        B = ruido_B
        return logistic(A), logistic(B)
    
    elif estrutura_causal == {"A->B", "A->C"}:
        ruido_B = generate_logits(R, tipo_ruido)
        ruido_C = generate_logits(R, tipo_ruido)

        A = generate_logits(R, tipo_ruido)
        B = 0.5 * A + ruido_B
        C = -0.5 * A + ruido_C
        return logistic(A), logistic(B), logistic(C)

    elif estrutura_causal == {"A->B", "A->C", "B->C", "C->D"}:
        ruido_A = generate_logits(R, tipo_ruido)
        ruido_B = generate_logits(R, tipo_ruido)
        ruido_C = generate_logits(R, tipo_ruido)
        ruido_D = generate_logits(R, tipo_ruido)

        A = ruido_A
        B = 0.5 * A + ruido_B
        C = -0.5 * A + 0.5 * B + ruido_C
        D = 0.5 * C + ruido_D
        return logistic(A), logistic(B), logistic(C), logistic(D)
    
    return None


def param_er(logit): return logit
def graph_gen_er(n, p): 
    return ig.Graph.Erdos_Renyi(n=n, p=p, directed=False)


def param_ge(logit): return logit
def graph_gen_geo(n, r): 
    return ig.Graph.GRG(n, radius=r)


def param_kr(logit): return (logit * 10).astype(int)
def graph_gen_kreg(n, k):
    try: 
        return ig.Graph.K_Regular(n, k)
    except: 
        return ig.Graph()


def param_ba(logit): return 2 * logit
def graph_gen_ba(n, p): 
    return ig.Graph.Barabasi(n=n, m=1, power=p, directed=False)

def param_ws(logit): return logit
def graph_gen_ws(n, p_w): 
    return ig.Graph.Watts_Strogatz(dim=1, size=n, nei=3, p=p_w)


def get_lam_and_evc(graph):
    if graph.ecount() == 0: return 0.0, 0.0
    try:
        evc, lam = graph.eigenvector_centrality(return_eigenvalue=True)
        return float(lam), float(np.mean(evc))
    except:
        adj_matrix = np.array(graph.get_adjacency().data)
        eigenvalues, eigenvectors = np.linalg.eigh(adj_matrix)
        max_idx = np.argmax(np.abs(eigenvalues))
        return float(np.abs(eigenvalues[max_idx])), float(np.mean(np.abs(eigenvectors[:, max_idx])))


def extract_graph_features(graph):
    n = graph.vcount()
    tc = graph.transitivity_undirected(mode="nan")
    if np.isnan(tc): 
        tc = 0.0
    bc = np.mean(graph.betweenness(directed=False)).item()
    cc = np.mean(graph.closeness(mode="all", normalized=True)).item()
    if np.isnan(cc): 
        cc = 0.0
    dc = (np.mean(graph.degree()) / (n - 1)).item() if n > 1 else 0.0
    
    try:
        as_coef = graph.assortativity_degree(directed=False)
        if np.isnan(as_coef): as_coef = 0.0
    except: as_coef = 0.0
        
    sr, evc = get_lam_and_evc(graph)
    return {r'$\lambda$': sr, 'TC': tc, 'BC': bc, 'CC': cc, 'EVC': evc, 'DC': dc, 'AS': as_coef}




def preprocessamento(df):
    df = df.rank(pct=True)
    return df




def gerar_amostras_grafos(modelo, n_v, R, *logits):
    # Dicionário mapeador para os modelos geradores e seus respectivos hiperparâmetros
    MODELOS_BASE = {
        "erdos_renyi": (graph_gen_er, param_er, "p"),
        "barabasi_albert": (graph_gen_ba, param_ba, "p"),
        "geometric": (graph_gen_geo, param_ge, "r"),
        "k_regular": (graph_gen_kreg, param_kr, "k"),
        "watts_strogatz": (graph_gen_ws, param_ws, "p_w")
    }

    # Tratamento para o novo caso com 4 vértices (A, B, C, D) usando o mesmo modelo
    if len(logits) == 4:
        if modelo not in MODELOS_BASE:
            raise ValueError(f"Modelo gerador desconhecido para 4 vértices homogêneos: {modelo}")
        
        gen_func, param_func, param_name = MODELOS_BASE[modelo]
        
        # Cria listas separadas para os 4 vértices
        listas = [[] for _ in range(4)]
        
        for j in range(R):
            for i in range(4):
                # Calcula o parâmetro correspondente ao logit do vértice atual
                param_val = param_func(logits[i][j])
                # Gera o grafo chamando a função com o nome correto do parâmetro
                G = gen_func(n=n_v, **{param_name: param_val})
                # Extrai features e armazena na lista correspondente
                listas[i].append(extract_graph_features(G))
                
        return pd.DataFrame(listas[0]), pd.DataFrame(listas[1]), pd.DataFrame(listas[2]), pd.DataFrame(listas[3])

    # Preserva o comportamento original para 2 vértices
    elif len(logits) == 2:
        A_logits, B_logits = logits[0], logits[1]
        lista_A, lista_B = [], []
        
        for j in range(R):
            # Modelos Base
            if modelo == "erdos_renyi": G_A = graph_gen_er(n=n_v, p=param_er(A_logits[j])); G_B = graph_gen_er(n=n_v, p=param_er(B_logits[j]))
            elif modelo == "barabasi_albert": G_A = graph_gen_ba(n=n_v, p=param_ba(A_logits[j])); G_B = graph_gen_ba(n=n_v, p=param_ba(B_logits[j]))
            elif modelo == "geometric": G_A = graph_gen_geo(n=n_v, r=param_ge(A_logits[j])); G_B = graph_gen_geo(n=n_v, r=param_ge(B_logits[j]))
            elif modelo == "k_regular": G_A = graph_gen_kreg(n=n_v, k=param_kr(A_logits[j])); G_B = graph_gen_kreg(n=n_v, k=param_kr(B_logits[j]))
            elif modelo == "watts_strogatz": G_A = graph_gen_ws(n=n_v, p_w=param_ws(A_logits[j])); G_B = graph_gen_ws(n=n_v, p_w=param_ws(B_logits[j]))
            
            # Combinações
            elif modelo == "barabasi_albert-erdos_renyi": G_A = graph_gen_ba(n=n_v, p=param_ba(A_logits[j])); G_B = graph_gen_er(n=n_v, p=param_er(B_logits[j]))
            elif modelo == "barabasi_albert-geometric": G_A = graph_gen_ba(n=n_v, p=param_ba(A_logits[j])); G_B = graph_gen_geo(n=n_v, r=param_ge(B_logits[j]))
            elif modelo == "barabasi_albert-k_regular": G_A = graph_gen_ba(n=n_v, p=param_ba(A_logits[j])); G_B = graph_gen_kreg(n=n_v, k=param_kr(B_logits[j]))
            elif modelo == "barabasi_albert-watts_strogatz": G_A = graph_gen_ba(n=n_v, p=param_ba(A_logits[j])); G_B = graph_gen_ws(n=n_v, p_w=param_ws(B_logits[j]))
            elif modelo == "erdos_renyi-geometric": G_A = graph_gen_er(n=n_v, p=param_er(A_logits[j])); G_B = graph_gen_geo(n=n_v, r=param_ge(B_logits[j]))
            elif modelo == "erdos_renyi-k_regular": G_A = graph_gen_er(n=n_v, p=param_er(A_logits[j])); G_B = graph_gen_kreg(n=n_v, k=param_kr(B_logits[j]))
            elif modelo == "erdos_renyi-watts_strogatz": G_A = graph_gen_er(n=n_v, p=param_er(A_logits[j])); G_B = graph_gen_ws(n=n_v, p_w=param_ws(B_logits[j]))
            elif modelo == "geometric-k_regular": G_A = graph_gen_geo(n=n_v, r=param_ge(A_logits[j])); G_B = graph_gen_kreg(n=n_v, k=param_kr(B_logits[j]))
            elif modelo == "geometric-watts_strogatz": G_A = graph_gen_geo(n=n_v, r=param_ge(A_logits[j])); G_B = graph_gen_ws(n=n_v, p_w=param_ws(B_logits[j]))
            elif modelo == "k_regular-watts_strogatz": G_A = graph_gen_kreg(n=n_v, k=param_kr(A_logits[j])); G_B = graph_gen_ws(n=n_v, p_w=param_ws(B_logits[j]))
            else:
                raise ValueError(f"Modelo gerador desconhecido: {modelo}")
              
            lista_A.append(extract_graph_features(G_A))
            lista_B.append(extract_graph_features(G_B))
            
        return pd.DataFrame(lista_A), pd.DataFrame(lista_B)

    # Preserva o comportamento original para 3 vértices
    elif len(logits) == 3:
        A_logits, B_logits, C_logits = logits[0], logits[1], logits[2]
        lista_A, lista_B, lista_C = [], [], []
        
        for j in range(R):
            # Modelos Base (Puros)
            if modelo == "erdos_renyi": G_A = graph_gen_er(n=n_v, p=param_er(A_logits[j])); G_B = graph_gen_er(n=n_v, p=param_er(B_logits[j])); G_C = graph_gen_er(n=n_v, p=param_er(C_logits[j]))
            elif modelo == "barabasi_albert": G_A = graph_gen_ba(n=n_v, p=param_ba(A_logits[j])); G_B = graph_gen_ba(n=n_v, p=param_ba(B_logits[j])); G_C = graph_gen_ba(n=n_v, p=param_ba(C_logits[j]))
            elif modelo == "geometric": G_A = graph_gen_geo(n=n_v, r=param_ge(A_logits[j])); G_B = graph_gen_geo(n=n_v, r=param_ge(B_logits[j])); G_C = graph_gen_geo(n=n_v, r=param_ge(C_logits[j]))
            elif modelo == "k_regular": G_A = graph_gen_kreg(n=n_v, k=param_kr(A_logits[j])); G_B = graph_gen_kreg(n=n_v, k=param_kr(B_logits[j])); G_C = graph_gen_kreg(n=n_v, k=param_kr(C_logits[j]))
            elif modelo == "watts_strogatz": G_A = graph_gen_ws(n=n_v, p_w=param_ws(A_logits[j])); G_B = graph_gen_ws(n=n_v, p_w=param_ws(B_logits[j])); G_C = graph_gen_ws(n=n_v, p_w=param_ws(C_logits[j]))
            
            # Combinações Mistas
            elif modelo == "barabasi_albert-barabasi_albert-erdos_renyi": G_A = graph_gen_ba(n=n_v, p=param_ba(A_logits[j])); G_B = graph_gen_ba(n=n_v, p=param_ba(B_logits[j])); G_C = graph_gen_er(n=n_v, p=param_er(C_logits[j]))
            elif modelo == "barabasi_albert-barabasi_albert-geometric": G_A = graph_gen_ba(n=n_v, p=param_ba(A_logits[j])); G_B = graph_gen_ba(n=n_v, p=param_ba(B_logits[j])); G_C = graph_gen_geo(n=n_v, r=param_ge(C_logits[j]))
            elif modelo == "barabasi_albert-barabasi_albert-k_regular": G_A = graph_gen_ba(n=n_v, p=param_ba(A_logits[j])); G_B = graph_gen_ba(n=n_v, p=param_ba(B_logits[j])); G_C = graph_gen_kreg(n=n_v, k=param_kr(C_logits[j]))
            elif modelo == "barabasi_albert-barabasi_albert-watts_strogatz": G_A = graph_gen_ba(n=n_v, p=param_ba(A_logits[j])); G_B = graph_gen_ba(n=n_v, p=param_ba(B_logits[j])); G_C = graph_gen_ws(n=n_v, p_w=param_ws(C_logits[j]))
            elif modelo == "erdos_renyi-erdos_renyi-geometric": G_A = graph_gen_er(n=n_v, p=param_er(A_logits[j])); G_B = graph_gen_er(n=n_v, p=param_er(B_logits[j])); G_C = graph_gen_geo(n=n_v, r=param_ge(C_logits[j]))
            elif modelo == "erdos_renyi-erdos_renyi-k_regular": G_A = graph_gen_er(n=n_v, p=param_er(A_logits[j])); G_B = graph_gen_er(n=n_v, p=param_er(B_logits[j])); G_C = graph_gen_kreg(n=n_v, k=param_kr(C_logits[j]))
            elif modelo == "erdos_renyi-erdos_renyi-watts_strogatz": G_A = graph_gen_er(n=n_v, p=param_er(A_logits[j])); G_B = graph_gen_er(n=n_v, p=param_er(B_logits[j])); G_C = graph_gen_ws(n=n_v, p_w=param_ws(C_logits[j]))
            elif modelo == "geometric-geometric-k_regular": G_A = graph_gen_geo(n=n_v, r=param_ge(A_logits[j])); G_B = graph_gen_geo(n=n_v, r=param_ge(B_logits[j])); G_C = graph_gen_kreg(n=n_v, k=param_kr(C_logits[j]))
            elif modelo == "geometric-geometric-watts_strogatz": G_A = graph_gen_geo(n=n_v, r=param_ge(A_logits[j])); G_B = graph_gen_geo(n=n_v, r=param_ge(B_logits[j])); G_C = graph_gen_ws(n=n_v, p_w=param_ws(C_logits[j]))
            elif modelo == "k_regular-k_regular-watts_strogatz": G_A = graph_gen_kreg(n=n_v, k=param_kr(A_logits[j])); G_B = graph_gen_kreg(n=n_v, k=param_kr(B_logits[j])); G_C = graph_gen_ws(n=n_v, p_w=param_ws(C_logits[j]))
            elif modelo == "barabasi_albert-k_regular-watts_strogatz": G_A = graph_gen_ba(n=n_v, p=param_ba(A_logits[j])); G_B = graph_gen_kreg(n=n_v, k=param_kr(B_logits[j])); G_C = graph_gen_ws(n=n_v, p_w=param_ws(C_logits[j]))
            else:
                raise ValueError(f"Modelo gerador desconhecido: {modelo}")
                
            lista_A.append(extract_graph_features(G_A))
            lista_B.append(extract_graph_features(G_B))
            lista_C.append(extract_graph_features(G_C))
            
        return pd.DataFrame(lista_A), pd.DataFrame(lista_B), pd.DataFrame(lista_C)

    else:
        raise ValueError(f"Número não suportado de entradas em logits: {len(logits)}")

### Cenario 1 e Cenario 2

In [12]:
def get_nome_modelos_AB():
    lista_unica = ["barabasi_albert", "watts_strogatz", "geometric", "erdos_renyi", "k_regular"]

    combinacoes_mistas = [
        "barabasi_albert-erdos_renyi",
        "barabasi_albert-geometric",
        "barabasi_albert-k_regular",
        "barabasi_albert-watts_strogatz",
        "erdos_renyi-geometric",
        "erdos_renyi-k_regular",
        "erdos_renyi-watts_strogatz",
        "geometric-k_regular",
        "geometric-watts_strogatz",
        "k_regular-watts_strogatz"
    ]

    return lista_unica + combinacoes_mistas



def simulacao_AB(modelos,
                 features, 
                 vertices_list,
                 tipo_ruido,
                 N_SIMS,
                 R,
                 estrutura,
                 caso,
                 save=False
                 ):
    
    
    import os
    import warnings
    from tqdm import tqdm
    warnings.filterwarnings("ignore", category=RuntimeWarning, message=".*eigenvector centralities.*")
    
    cenarios = list(product(
        modelos, 
        tipo_ruido, 
        vertices_list, 
        R))
    resultados_power = []
    lista_p_valores = [] 
    lista_metricas = []
    for (modelo, ruido, n_vert, r) in tqdm(cenarios):
        sucessos_rejeicao = {
                    "A_B_taxa1%": {feat: 0 for feat in features},
                    "A_B_taxa5%": {feat: 0 for feat in features},
                    "A_B_taxa10%": {feat: 0 for feat in features}}
        
        for sim in range(N_SIMS):
            tentativas = 0
            sucesso_geracao = False

            while tentativas < 20:
                try:
                    A, B = gerar_estrutura_causal_logits(r, ruido, estrutura)
                    df_A, df_B = gerar_amostras_grafos(modelo, n_vert, r, A, B)
                    sucesso_geracao = True
                    break
                
                except Exception as e:
                    tentativas += 1
                    continue

            if not sucesso_geracao:
                raise RuntimeError("Travamento: O gerador falhou 20 vezes seguidas.")

            for feature in features:
                safe_feat_name = feature.replace(r"$\lambda$", "Lambda").replace("$", "").replace("\\", "").replace("_", "")
                col_A = f"{safe_feat_name}_A"
                col_B = f"{safe_feat_name}_B"
                df_merged = pd.concat([df_A[feature].rename(col_A), df_B[feature].rename(col_B)], axis=1)
            
                if df_merged[col_A].std() < 1e-9 or df_merged[col_B].std() < 1e-9:
                    lista_p_valores.append({
                                            "Modelo": modelo, 
                                            "Vertices": n_vert,
                                            "ruido":ruido, 
                                            "R": r,      
                                            "Feature": feature, 
                                            "Direcao": "A->B", 
                                            "Simulacao": sim, 
                                            "p_valor": None
                                        })
                    lista_metricas.append({
                                            "Caso":caso,
                                            "estrutura":estrutura,
                                            "Modelo": modelo, 
                                             "Vertices": n_vert,
                                              "ruido":ruido, 
                                              "R": r,      
                                              "Feature": feature, 
                                              "Simulacao": sim, 
                                              "RMSEA": None,
                                              "AIC": None,
                                              "BIC": None,
                                              "chi2":None,
                                              "chi2_p_val":None
                                                            })
                    continue

                df_merged = preprocessamento(df_merged)
                modelos_sem = {
                                    "A->B": {"formula": f"{col_B} ~ {col_A}", "lval": col_B, "rval": col_A},
                                }
                
                for direcao, detalhes in modelos_sem.items():
                    p_value, rmsea, aic, bic, chi2, chi2_p_val = None, None, None, None, None, None                    
                    try:
                        model_sem = semopy.Model(detalhes["formula"])
                        model_sem.fit(df_merged)
                        ins = model_sem.inspect()  
                        metricas = semopy.calc_stats(model_sem)
                        chi2 = metricas["chi2"].values[0]
                        chi2_p_val = metricas["chi2 p-value"].values[0]
                        rmsea = metricas["RMSEA"].values[0]
                        aic = metricas["AIC"].values[0]
                        bic = metricas["BIC"].values[0]

                

                        row = ins[(ins['lval'] == detalhes["lval"]) & (ins['rval'] == detalhes["rval"])]
                                            
                        if not row.empty:
                            pval_raw = row['p-value'].values[0]
                            p_value = float(pval_raw.replace('<', '').strip()) if isinstance(pval_raw, str) else float(pval_raw)                   

                        if p_value is not None and p_value < 0.10:
                            sucessos_rejeicao["A_B_taxa10%"][feature] += 1

                        if p_value is not None and p_value < 0.05:
                            sucessos_rejeicao["A_B_taxa5%"][feature] += 1

                        if p_value is not None and p_value < 0.01:
                            sucessos_rejeicao["A_B_taxa1%"][feature] += 1
                              
                    except Exception as e:
                        #print(e)
                        pass

                    lista_p_valores.append({
                                            "Modelo": modelo,
                                            "Vertices": n_vert,
                                            "ruido":ruido,
                                            "R":r,
                                            "Feature": feature,
                                            "Direcao": "A->B",
                                            "Simulacao": sim,
                                            "p_valor": p_value
                                        })
                    lista_metricas.append({
                                           "Caso":caso,
                                           "estrutura":estrutura,
                                            "Modelo": modelo, 
                                            "Vertices": n_vert,
                                             "ruido":ruido, 
                                            "R": r,      
                                            "Feature": feature, 
                                           "Simulacao": sim,
                                           "chi2": chi2,
                                           "chi2_p_val":chi2_p_val,
                                           "RMSEA": rmsea,
                                           "AIC": aic,
                                            "BIC": bic
                                          })

                    

        for feature in features:
            poder_A_B_1 = sucessos_rejeicao["A_B_taxa1%"][feature] / N_SIMS
            poder_A_B_5 = sucessos_rejeicao["A_B_taxa5%"][feature] / N_SIMS
            poder_A_B_10 = sucessos_rejeicao["A_B_taxa10%"][feature] / N_SIMS

            resultados_power.append({
                        "Modelo": modelo, 
                        "Vértices": n_vert,
                        "ruido":ruido,
                        "R":r, 
                        "Feature": feature,
                        "Direcao": "A->B", 
                        "Poder_1%": poder_A_B_1,
                        "Poder_5%": poder_A_B_5,
                        "Poder_10%": poder_A_B_10
                    })

    

    titulos_formatados = {
        # Modelos Simples
        "erdos_renyi": "Erdös-Rényi",
        "geometric": "Geometric",
        "k_regular": "K-Regular",
        "barabasi_albert": "Barabási-Albert",
        "watts_strogatz": "Watts-Strogatz",
        
        # Modelos Mistos (Mantendo por extenso)
        "barabasi_albert-erdos_renyi": "Barabási-Albert + Erdös-Rényi",
        "barabasi_albert-geometric": "Barabási-Albert + Geometric",
        "barabasi_albert-k_regular": "Barabási-Albert + K-Regular",
        "barabasi_albert-watts_strogatz": "Barabási-Albert + Watts-Strogatz",
        "erdos_renyi-geometric": "Erdös-Rényi + Geometric",
        "erdos_renyi-k_regular": "Erdös-Rényi + K-Regular",
        "erdos_renyi-watts_strogatz": "Erdös-Rényi + Watts-Strogatz",
        "geometric-k_regular": "Geometric + K-Regular",
        "geometric-watts_strogatz": "Geometric + Watts-Strogatz",
        "k_regular-watts_strogatz": "K-Regular + Watts-Strogatz"
    }

    df_resultados = pd.DataFrame(resultados_power) 
    df_p_valores = pd.DataFrame(lista_p_valores)
    df_metricas = pd.DataFrame(lista_metricas)

    if save:
        try:
            base_dir = os.path.dirname(os.path.abspath(__file__))
        except NameError:
            base_dir = os.getcwd()
        path_dir = os.path.abspath(os.path.join(base_dir, "..", "dados", "resultados", "sim", caso))
        os.makedirs(path_dir, exist_ok=True)

        mapa_titulos = {mod: titulos_formatados.get(mod, mod) for mod in modelos}
        df_resultados['Titulo_Subplot'] = df_resultados['Modelo'].map(mapa_titulos)

        df_resultados.to_csv(os.path.join(path_dir, "resultados_power.csv"), index=False)
        df_p_valores.to_csv(os.path.join(path_dir, "p_valores.csv"), index=False)
        df_metricas.to_csv(os.path.join(path_dir, "metrics.csv"), index=False)

    return df_resultados, df_p_valores, df_metricas

In [13]:
# n_sims = 1000
# R = [20, 50, 100]
# vertices_list = [20, 60, 100]
# modelos = get_nome_modelos_AB()
# features = [r"$\lambda$", "TC", "BC", "CC", "EVC", "DC", "AS"]
# tipo_ruido = ["uniform", "gaussian", "chi_square", "F"]


# # # Parâmetros para o Caso 1
# params_caso1 = {
#     "modelos": modelos,
#     "features": features,
#     "vertices_list": vertices_list,
#     "tipo_ruido": tipo_ruido,
#     "N_SIMS": n_sims,
#     "R": R,

#     "estrutura": {"A<->B"},
#     "caso": "Caso_1",
#     "save": True
# }

# # # Parâmetros para o Caso 2
# params_caso2 = {
#     "modelos": modelos,
#     "features": features,
#     "vertices_list": vertices_list,
#     "tipo_ruido": tipo_ruido,
#     "N_SIMS": n_sims,
#     "R": R,
    
#     "estrutura": {"A->B"},
#     "caso": "Caso_2",
#     "save": True
# }

# # Cria uma lista com as configurações
# lista_de_casos = [params_caso1, params_caso2]

# # Executa a função em paralelo usando 2 workers (n_jobs=2)
# resultados = Parallel(n_jobs=2)(
#     delayed(simulacao_AB)(**kwargs) for kwargs in lista_de_casos
# )

# # Desempacotando as tuplas retornadas
# (df_res_1, df_pval_1), (df_res_2, df_pval_2) = resultados

### Cenario 3 - Confundidor Real

In [14]:
import os
import re
import warnings
from itertools import product
import pandas as pd
import semopy
from tqdm import tqdm


def extrair_arestas_da_formula(formula_str):
    """Extrai os pares (rval, lval) correspondentes às regressões 'lval ~ rval' na fórmula SEM."""
    arestas = []
    # Remove chaves de formatação, se presentes ex: "{B} ~ {A}" -> "B ~ A"
    formula_limpa = formula_str.replace("{", "").replace("}", "")

    for linha in formula_limpa.strip().split("\n"):
        linha = linha.strip()
        if "~" in linha and not linha.startswith("#"):
            partes = linha.split("~")
            lval = partes[0].strip()
            rvals = [r.strip() for r in partes[1].split("+")]
            for rval in rvals:
                arestas.append((rval.strip(), lval.strip()))
    return arestas


def get_nome_modelos_ABC():
    return [
        "barabasi_albert",
        "watts_strogatz",
        "geometric",
        "erdos_renyi",
        "k_regular",
    ]


def simulacao_ABC(
    modelos,
    features,
    vertices_list,
    tipo_ruido,
    N_SIMS,
    R,
    estrutura,
    caso,
    formula_sem_template="{B} ~ {A}\n{C} ~ {A} + {B}",
    save=False
):

    warnings.filterwarnings(
        "ignore",
        category=RuntimeWarning,
        message=".*eigenvector centralities.*",
    )

    # Identifica as arestas genéricas a serem testadas com base no template recebido (ex: [('A','B'), ('A','C'), ('B','C')])
    arestas_base = extrair_arestas_da_formula(formula_sem_template)

    cenarios = list(product(modelos, tipo_ruido, vertices_list, R))

    resultados_power = []
    lista_p_valores = []
    lista_metricas = []

    for modelo, ruido, n_v, r in tqdm(cenarios, disable=True):

        # Inicializa dinamicamente as métricas de rejeição para cada aresta definida na fórmula
        sucessos_rejeicao = {}
        for rval_b, lval_b in arestas_base:
            chave_aresta = f"{rval_b}_{lval_b}"
            for taxa in ["1%", "5%", "10%"]:
                sucessos_rejeicao[f"{chave_aresta}_taxa{taxa}"] = {
                    feat: 0 for feat in features
                }

        for sim in range(N_SIMS):
            tentativas = 0
            sucesso_geracao = False

            while tentativas < 20:
                try:
                    A, B, C = gerar_estrutura_causal_logits(
                        r, ruido, estrutura
                    )
                    df_A, df_B, df_C = gerar_amostras_grafos(
                        modelo, n_v, r, A, B, C
                    )
                    sucesso_geracao = True
                    break
                except Exception as e:
                    tentativas += 1
                    continue

            if not sucesso_geracao:
                raise RuntimeError("Travamento: O gerador falhou 20 vezes seguidas.")
            
            for feature in features:
                safe_feat_name = (
                    feature.replace(r"$\lambda$", "Lambda")
                    .replace("$", "")
                    .replace("\\", "")
                    .replace("_", "")
                )
                col_A = f"{safe_feat_name}_A"
                col_B = f"{safe_feat_name}_B"
                col_C = f"{safe_feat_name}_C"

                # Concatenação das 3 variáveis
                df_merged = pd.concat(
                    [
                        df_A[feature].reset_index(drop=True).rename(col_A),
                        df_B[feature].reset_index(drop=True).rename(col_B),
                        df_C[feature].reset_index(drop=True).rename(col_C),
                    ],
                    axis=1,
                ).dropna()

                # Trava de variância para todas as colunas envolvidas
                variancia_insuficiente = any(
                    df_merged[col].std() < 1e-9 for col in df_merged.columns
                )
                if (len(df_merged) < 5) or variancia_insuficiente:
                    for rval_b, lval_b in arestas_base:
                        lista_p_valores.append({
                            "Modelo": modelo,
                            "Vertices": n_v,
                            "ruido": ruido,
                            "R": r,
                            "Feature": feature,
                            "Aresta": f"{rval_b}->{lval_b}",
                            "Simulacao": sim,
                            "p_valor": None,
                        })
                    lista_metricas.append({
                            "Caso":caso,
                            "estrutura":estrutura,
                            "Modelo": modelo, 
                                "Vertices": n_v,
                                "ruido":ruido, 
                                "R": r,      
                                "Feature": feature, 
                                "Simulacao": sim, 
                                "RMSEA": None,
                                "AIC": None,
                                "BIC": None,
                                "chi2":None,
                                "chi2_p_val":None
                                            })
                    continue

                df_merged = preprocessamento(df_merged)

                # Dicionário mapeando os nós abstratos da fórmula para os nomes reais de colunas
                mapeamento_cols = {"A": col_A, "B": col_B, "C": col_C}

                # Substitui as variáveis no template da fórmula pelos nomes formatados das colunas
                formula_sem_atual = formula_sem_template.format(
                    **mapeamento_cols
                )

                # Inicializa p-valores como None para todas as arestas
                p_vals_simulacao = {
                    f"{rval}->{lval}": None for rval, lval in arestas_base
                }
                
                rmsea, aic, bic, chi2, chi2_p_val = None, None, None, None, None  

                try:
                    model_sem = semopy.Model(formula_sem_atual)
                    model_sem.fit(df_merged)
                    ins = model_sem.inspect()
                    metricas = semopy.calc_stats(model_sem)
                    chi2 = metricas["chi2"].values[0]
                    chi2_p_val = metricas["chi2 p-value"].values[0]
                    rmsea = metricas["RMSEA"].values[0]
                    aic = metricas["AIC"].values[0]
                    bic = metricas["BIC"].values[0]

                    for rval_b, lval_b in arestas_base:
                        lval_real = mapeamento_cols[lval_b]
                        rval_real = mapeamento_cols[rval_b]
                        nome_aresta_format = f"{rval_b}->{lval_b}"
                        nome_aresta_chave = f"{rval_b}_{lval_b}"

                        # O operador '~' garante que estamos olhando para regressões
                        row = ins[
                            (ins["lval"] == lval_real)
                            & (ins["rval"] == rval_real)
                            & (ins["op"] == "~")
                        ]

                        if not row.empty:
                            pval_raw = row["p-value"].values[0]
                            p_value = (
                                float(pval_raw.replace("<", "").strip())
                                if isinstance(pval_raw, str)
                                else float(pval_raw)
                            )
                            p_vals_simulacao[nome_aresta_format] = p_value

                            if p_value < 0.01:
                                sucessos_rejeicao[
                                    f"{nome_aresta_chave}_taxa1%"
                                ][feature] += 1
                            if p_value < 0.05:
                                sucessos_rejeicao[
                                    f"{nome_aresta_chave}_taxa5%"
                                ][feature] += 1
                            if p_value < 0.10:
                                sucessos_rejeicao[
                                    f"{nome_aresta_chave}_taxa10%"
                                ][feature] += 1

                except Exception:
                    pass  # Falha de convergência do SEM: mantém os p_vals como None

                # Registra o p-valor obtido em cada aresta da simulação
                lista_metricas.append({
                            "Caso":caso,
                            "estrutura":estrutura,
                            "Modelo": modelo, 
                                "Vertices": n_v,
                                "ruido":ruido, 
                                "R": r,      
                                "Feature": feature, 
                                "Simulacao": sim, 
                                "RMSEA": rmsea,
                                "AIC": aic,
                                "BIC": bic,
                                "chi2":chi2,
                                "chi2_p_val":chi2_p_val
                                            })
                
                for rval_b, lval_b in arestas_base:
                    aresta_fmt = f"{rval_b}->{lval_b}"
                    lista_p_valores.append({
                        "Modelo": modelo,
                        "Vertices": n_v,
                        "ruido": ruido,
                        "R": r,
                        "Feature": feature,
                        "Aresta": aresta_fmt,
                        "Simulacao": sim,
                        "p_valor": p_vals_simulacao[aresta_fmt],
                    })


        # Consolidando o poder estatístico de cada aresta
        for feature in features:
            for rval_b, lval_b in arestas_base:
                chave_aresta = f"{rval_b}_{lval_b}"
                poder_1 = (
                    sucessos_rejeicao[f"{chave_aresta}_taxa1%"][feature]
                    / N_SIMS
                )
                poder_5 = (
                    sucessos_rejeicao[f"{chave_aresta}_taxa5%"][feature]
                    / N_SIMS
                )
                poder_10 = (
                    sucessos_rejeicao[f"{chave_aresta}_taxa10%"][feature]
                    / N_SIMS
                )

                resultados_power.append({
                    "Modelo": modelo,
                    "Vértices": n_v,
                    "ruido": ruido,
                    "R": r,
                    "Feature": feature,
                    "Direcao": f"{rval_b}->{lval_b}",
                    "Poder_1%": poder_1,
                    "Poder_5%": poder_5,
                    "Poder_10%": poder_10,
                })

    

    df_resultados = pd.DataFrame(resultados_power)
    df_p_valores = pd.DataFrame(lista_p_valores)
    df_metricas = pd.DataFrame(lista_metricas)

    if save:
        try:
                base_dir = os.path.dirname(os.path.abspath(__file__))
        except NameError:
            base_dir = os.getcwd()
    
        path_dir = os.path.abspath(
            os.path.join(base_dir, "..", "dados", "resultados", "sim", caso)
        )
        os.makedirs(path_dir, exist_ok=True)
    
        titulos_formatados = {
            "erdos_renyi": "Erdös-Rényi",
            "geometric": "Geometric",
            "k_regular": "K-Regular",
            "barabasi_albert": "Barabási-Albert",
            "watts_strogatz": "Watts-Strogatz",
        }
        
        mapa_titulos = {mod: titulos_formatados.get(mod, mod) for mod in modelos}
        df_resultados["Titulo_Subplot"] = df_resultados["Modelo"].map(mapa_titulos)

        df_resultados.to_csv(
            os.path.join(path_dir, "resultados_power.csv"), index=False
        )
        df_p_valores.to_csv(
            os.path.join(path_dir, "p_valores.csv"), index=False
        )

        df_metricas.to_csv(
            os.path.join(path_dir, "metrics.csv"), index=False
        )

    return df_resultados, df_p_valores, df_metricas

In [15]:
dags_classes_equivalencia = {
    # =========================================================
    # CLASSE 1: Esqueleto com A no meio (Nenhum colisor)
    # =========================================================
    "SEM_Class_1_Garfo_A_REAL": "{B} ~ {A}\n{C} ~ {A}", # Garfo em A (B <- A -> C)
    "SEM_Class_1_Cadeia_B_A_C": "{A} ~ {B}\n{C} ~ {A}", # Cadeia B -> A -> C
    "SEM_Class_1_Cadeia_C_A_B": "{A} ~ {C}\n{B} ~ {A}", # Cadeia C -> A -> B
    
    # =========================================================
    # CLASSE 2: Esqueleto com A no meio (Colisor em A)
    # =========================================================
    "SEM_Class_2_Colisor_A": "{A} ~ {B} + {C}", # Colisor em A (B -> A <- C)

    # =========================================================
    # CLASSE 3: Esqueleto com B no meio (Nenhum colisor)
    # =========================================================
    "SEM_Class_3_Cadeia_A_B_C": "{B} ~ {A}\n{C} ~ {B}", # Cadeia A -> B -> C (sua "Cadeia_B")
    "SEM_Class_3_Cadeia_C_B_A": "{B} ~ {C}\n{A} ~ {B}", # Cadeia C -> B -> A
    "SEM_Class_3_Garfo_B": "{A} ~ {B}\n{C} ~ {B}",      # Garfo em B (A <- B -> C)
    
    # =========================================================
    # CLASSE 4: Esqueleto com B no meio (Colisor em B)
    # =========================================================
    "SEM_Class_4_Colisor_B": "{B} ~ {A} + {C}", # Colisor em B (A -> B <- C)
    
    # =========================================================
    # CLASSE 5: Esqueleto com C no meio (Nenhum colisor)
    # =========================================================
    "SEM_Class_5_Cadeia_A_C_B": "{C} ~ {A}\n{B} ~ {C}", # Cadeia A -> C -> B (sua "Cadeia_C")
    "SEM_Class_5_Cadeia_B_C_A": "{C} ~ {B}\n{A} ~ {C}", # Cadeia B -> C -> A
    "SEM_Class_5_Garfo_C": "{A} ~ {C}\n{B} ~ {C}",      # Garfo em C (A <- C -> B)
  
    # =========================================================
    # CLASSE 6: Esqueleto com C no meio (Colisor em C)
    # =========================================================
    "SEM_Class_6_Colisor_C": "{C} ~ {A} + {B}", # Colisor em C (A -> C <- B)

    # =========================================================
    # CLASSE 7: Triângulos Completos (6 permutações acíclicas)
    # =========================================================
    "SEM_Class_7_Triangulo_A_B-B_C-A_C": "{B} ~ {A}\n{C} ~ {A} + {B}", # A->B, B->C, A->C (Fonte: A, Sumidouro: C)
    "SEM_Class_7_Triangulo_A_C-C_B-A_B": "{C} ~ {A}\n{B} ~ {A} + {C}", # A->C, C->B, A->B (Fonte: A, Sumidouro: B)
    "SEM_Class_7_Triangulo_B_A-A_C-B_C": "{A} ~ {B}\n{C} ~ {B} + {A}", # B->A, A->C, B->C (Fonte: B, Sumidouro: C)
    "SEM_Class_7_Triangulo_B_C-C_A-B_A": "{C} ~ {B}\n{A} ~ {B} + {C}", # B->C, C->A, B->A (Fonte: B, Sumidouro: A)
    "SEM_Class_7_Triangulo_C_A-A_B-C_B": "{A} ~ {C}\n{B} ~ {C} + {A}", # C->A, A->B, C->B (Fonte: C, Sumidouro: B)
    "SEM_Class_7_Triangulo_C_B-B_A-C_A": "{B} ~ {C}\n{A} ~ {C} + {B}", # C->B, B->A, C->A (Fonte: C, Sumidouro: A)
}

In [16]:
R = [20, 50, 100]  
vertices_list = [20, 60, 100]             
N_SIMS = 1000           
tipo_ruido = ["uniform", "gaussian", "chi_square", "F"]

estrutura = {"A->B", "A->C"} 
cenario = "Cenario_3"

modelos = get_nome_modelos_ABC()
features = [r"$\lambda$", "TC", "BC", "CC", "EVC", "DC", "AS"]

if __name__ == "__main__":
    resultados = Parallel(n_jobs=9)(
        delayed(simulacao_ABC)(
            modelos=modelos,
            features=features,
            vertices_list=vertices_list,
            tipo_ruido=tipo_ruido,
            N_SIMS=N_SIMS,
            R=R,
            estrutura=estrutura,
            caso=f"{cenario}/{nome_classe}",
            formula_sem_template=formula,
            save=True
        )
        for nome_classe, formula in dags_classes_equivalencia.items()
    )

    # Reorganiza o retorno
    resultados_por_classe = dict(
        zip(dags_classes_equivalencia.keys(), resultados)
    )

### Cenario 4 - Efeito direto e efeito indireto (4 vertices)

In [8]:
import os
import re
import warnings
from itertools import product
import pandas as pd
import semopy
from tqdm import tqdm


def extrair_arestas_da_formula(formula_str):
    """Extrai os pares (rval, lval) correspondentes às regressões 'lval ~ rval' na fórmula SEM."""
    arestas = []
    # Remove chaves de formatação, se presentes ex: "{B} ~ {A}" -> "B ~ A"
    formula_limpa = formula_str.replace("{", "").replace("}", "")

    for linha in formula_limpa.strip().split("\n"):
        linha = linha.strip()
        if "~" in linha and not linha.startswith("#"):
            partes = linha.split("~")
            lval = partes[0].strip()
            rvals = [r.strip() for r in partes[1].split("+")]
            for rval in rvals:
                arestas.append((rval.strip(), lval.strip()))
    return arestas


def get_nome_modelos_ABCD():
    return [
        "barabasi_albert",
        "watts_strogatz",
        "geometric",
        "erdos_renyi",
        "k_regular",
    ]


def simulacao_ABCD(
    modelos,
    features,
    vertices_list,
    tipo_ruido,
    N_SIMS,
    R,
    estrutura,
    caso,
    formula_sem_template="{B} ~ {A}\n{C} ~ {A} + {B}\n{D} ~ {C}",  # <- Template estendido para 4 vértices
    save=False
):

    warnings.filterwarnings(
        "ignore",
        category=RuntimeWarning,
        message=".*eigenvector centralities.*",
    )

    # Identifica as arestas genéricas a serem testadas com base no template recebido
    arestas_base = extrair_arestas_da_formula(formula_sem_template)

    cenarios = list(product(modelos, tipo_ruido, vertices_list, R))

    resultados_power = []
    lista_p_valores = []
    lista_metricas = []

    for modelo, ruido, n_v, r in tqdm(cenarios, disable=False):

        # Inicializa dinamicamente as métricas de rejeição para cada aresta definida na fórmula
        sucessos_rejeicao = {}
        for rval_b, lval_b in arestas_base:
            chave_aresta = f"{rval_b}_{lval_b}"
            for taxa in ["1%", "5%", "10%"]:
                sucessos_rejeicao[f"{chave_aresta}_taxa{taxa}"] = {
                    feat: 0 for feat in features
                }

        for sim in range(N_SIMS):
            tentativas = 0
            sucesso_geracao = False

            while tentativas < 20:
                try:
                    # Gerando logits e grafos para 4 vértices (A, B, C, D)
                    A, B, C, D = gerar_estrutura_causal_logits(
                        r, ruido, estrutura
                    )
                    df_A, df_B, df_C, df_D = gerar_amostras_grafos(
                        modelo, n_v, r, A, B, C, D
                    )
                    sucesso_geracao = True
                    break
                except Exception as e:
                    tentativas += 1
                    continue
            if not sucesso_geracao:
                raise RuntimeError("Travamento: O gerador falhou 20 vezes seguidas.") 

            for feature in features:
                safe_feat_name = (
                    feature.replace(r"$\lambda$", "Lambda")
                    .replace("$", "")
                    .replace("\\", "")
                    .replace("_", "")
                )
                col_A = f"{safe_feat_name}_A"
                col_B = f"{safe_feat_name}_B"
                col_C = f"{safe_feat_name}_C"
                col_D = f"{safe_feat_name}_D"

                # Concatenação das 4 variáveis
                df_merged = pd.concat(
                    [
                        df_A[feature].reset_index(drop=True).rename(col_A),
                        df_B[feature].reset_index(drop=True).rename(col_B),
                        df_C[feature].reset_index(drop=True).rename(col_C),
                        df_D[feature].reset_index(drop=True).rename(col_D),
                    ],
                    axis=1,
                ).dropna()

                # Trava de variância para todas as colunas envolvidas
                variancia_insuficiente = any(
                    df_merged[col].std() < 1e-9 for col in df_merged.columns
                )
                if (len(df_merged) < 5) or variancia_insuficiente:
                    lista_metricas.append({
                                "Caso":caso,
                                "estrutura":estrutura,
                                "Modelo": modelo, 
                                "Vertices": n_v,
                                "ruido":ruido, 
                                "R": r,      
                                "Feature": feature, 
                                "Simulacao": sim, 
                                "RMSEA": None,
                                "AIC": None,
                                "BIC": None,
                                "chi2":None,
                                "chi2_p_val":None
                            })
                    for rval_b, lval_b in arestas_base:
                        lista_p_valores.append({
                            "Modelo": modelo,
                            "Vertices": n_v,
                            "ruido": ruido,
                            "R": r,
                            "Feature": feature,
                            "Aresta": f"{rval_b}->{lval_b}",
                            "Simulacao": sim,
                            "p_valor": None,
                        })
                    continue

                df_merged = preprocessamento(df_merged)

                # Dicionário mapeando os nós abstratos para os nomes das colunas com 4 vértices
                mapeamento_cols = {
                    "A": col_A,
                    "B": col_B,
                    "C": col_C,
                    "D": col_D,
                }

                # Substitui as variáveis no template da fórmula pelos nomes formatados das colunas
                formula_sem_atual = formula_sem_template.format(
                    **mapeamento_cols
                )

                # Inicializa p-valores como None para todas as arestas
                p_vals_simulacao = {
                    f"{rval}->{lval}": None for rval, lval in arestas_base
                }
                rmsea, aic, bic, chi2, chi2_p_val = None, None, None, None, None

                try:
                    model_sem = semopy.Model(formula_sem_atual)
                    model_sem.fit(df_merged)
                    ins = model_sem.inspect()
                    metricas = semopy.calc_stats(model_sem)
                    chi2 = metricas["chi2"].values[0]
                    chi2_p_val = metricas["chi2 p-value"].values[0]
                    rmsea = metricas["RMSEA"].values[0]
                    aic = metricas["AIC"].values[0]
                    bic = metricas["BIC"].values[0]

                    for rval_b, lval_b in arestas_base:
                        lval_real = mapeamento_cols[lval_b]
                        rval_real = mapeamento_cols[rval_b]
                        nome_aresta_format = f"{rval_b}->{lval_b}"
                        nome_aresta_chave = f"{rval_b}_{lval_b}"

                        # O operador '~' garante que estamos a olhar para regressões
                        row = ins[
                            (ins["lval"] == lval_real)
                            & (ins["rval"] == rval_real)
                            & (ins["op"] == "~")
                        ]

                        if not row.empty:
                            pval_raw = row["p-value"].values[0]
                            p_value = (
                                float(pval_raw.replace("<", "").strip())
                                if isinstance(pval_raw, str)
                                else float(pval_raw)
                            )
                            p_vals_simulacao[nome_aresta_format] = p_value

                            if p_value < 0.01:
                                sucessos_rejeicao[
                                    f"{nome_aresta_chave}_taxa1%"
                                ][feature] += 1
                            if p_value < 0.05:
                                sucessos_rejeicao[
                                    f"{nome_aresta_chave}_taxa5%"
                                ][feature] += 1
                            if p_value < 0.10:
                                sucessos_rejeicao[
                                    f"{nome_aresta_chave}_taxa10%"
                                ][feature] += 1

                except Exception:
                    pass  # Falha de convergência do SEM: mantém os p_vals como None

                lista_metricas.append({
                    "Caso":caso,
                    "estrutura":estrutura,
                    "Modelo": modelo, 
                    "Vertices": n_v,
                    "ruido":ruido, 
                    "R": r,      
                    "Feature": feature, 
                    "Simulacao": sim, 
                    "RMSEA": rmsea,
                    "AIC": aic,
                    "BIC": bic,
                    "chi2":chi2,
                    "chi2_p_val":chi2_p_val
                    })
                
                # Regista o p-valor obtido em cada aresta da simulação
                for rval_b, lval_b in arestas_base:
                    aresta_fmt = f"{rval_b}->{lval_b}"
                    lista_p_valores.append({
                        "Modelo": modelo,
                        "Vertices": n_v,
                        "ruido": ruido,
                        "R": r,
                        "Feature": feature,
                        "Aresta": aresta_fmt,
                        "Simulacao": sim,
                        "p_valor": p_vals_simulacao[aresta_fmt],
                    })

        # Consolidando o poder estatístico de cada aresta
        for feature in features:
            for rval_b, lval_b in arestas_base:
                chave_aresta = f"{rval_b}_{lval_b}"
                poder_1 = (
                    sucessos_rejeicao[f"{chave_aresta}_taxa1%"][feature]
                    / N_SIMS
                )
                poder_5 = (
                    sucessos_rejeicao[f"{chave_aresta}_taxa5%"][feature]
                    / N_SIMS
                )
                poder_10 = (
                    sucessos_rejeicao[f"{chave_aresta}_taxa10%"][feature]
                    / N_SIMS
                )

                resultados_power.append({
                    "Modelo": modelo,
                    "Vértices": n_v,
                    "ruido": ruido,
                    "R": r,
                    "Feature": feature,
                    "Direcao": f"{rval_b}->{lval_b}",
                    "Poder_1%": poder_1,
                    "Poder_5%": poder_5,
                    "Poder_10%": poder_10,
                })
    
    
    df_resultados = pd.DataFrame(resultados_power)
    df_p_valores = pd.DataFrame(lista_p_valores)
    df_metricas = pd.DataFrame(lista_metricas)
    
    if save:
        try:
            base_dir = os.path.dirname(os.path.abspath(__file__))
        except NameError:
            base_dir = os.getcwd()

        path_dir = os.path.abspath(
            os.path.join(base_dir, "..", "dados", "resultados", "sim", caso)
        )
        os.makedirs(path_dir, exist_ok=True)

        titulos_formatados = {
            "erdos_renyi": "Erdös-Rényi",
            "geometric": "Geometric",
            "k_regular": "K-Regular",
            "barabasi_albert": "Barabási-Albert",
            "watts_strogatz": "Watts-Strogatz",
        }

        

        mapa_titulos = {mod: titulos_formatados.get(mod, mod) for mod in modelos}
        df_resultados["Titulo_Subplot"] = df_resultados["Modelo"].map(mapa_titulos)

        df_resultados.to_csv(
            os.path.join(path_dir, "resultados_power.csv"), index=False
        )
        df_p_valores.to_csv(
            os.path.join(path_dir, "p_valores.csv"), index=False
        )

        df_metricas.to_csv(
            os.path.join(path_dir, "metrics.csv"), index=False
        )

        

    return df_resultados, df_p_valores, df_metricas

In [ ]:
R = [20, 50, 100]  
vertices_list = [20, 60, 100]             
N_SIMS = 1000           
tipo_ruido = ["uniform", "gaussian", "chi_square", "F"]


modelos_sem = {
    "SEM_Grafo_REAL":{
        "formula": "{B} ~ {A}\n{C} ~ {A} + {B}\n{D} ~ {C}",
        "estrutura": ["A->B", "A->C", "B->C", "C->D"]}
    }

estrutura_real = {"A->B", "A->C", "B->C", "C->D"}
modelos = get_nome_modelos_ABCD()
features = [r"$\lambda$", "TC", "BC", "CC", "EVC", "DC", "AS"]


cenario = "Cenario_4"
for nome_modelo, valor in modelos_sem.items():
    df_r, df_p, df_m = simulacao_ABCD(
                modelos=modelos,
                features=features,
                vertices_list=vertices_list,
                tipo_ruido=tipo_ruido,
                N_SIMS=N_SIMS,
                R=R,
                estrutura=estrutura_real,
                caso=f"{cenario}/{nome_modelo}",
                formula_sem_template=valor["formula"],
                save=True
                )

  0%|          | 0/180 [00:00<?, ?it/s]

100%|██████████| 180/180 [12:42:37<00:00, 254.21s/it]  
